In [ ]:
!pip install -q \
langchain==0.1.16 \
langchain-community==0.0.34 \
chromadb==0.4.24 \
pypdf \
sentence-transformers \
langgraph

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langgraph.graph import StateGraph

import os

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
loader = PyPDFLoader(list(uploaded.keys())[0])
documents = loader.load()

print("Total Pages:", len(documents))
print("\nSample:\n", documents[0].page_content[:500])

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(chunks, embeddings)

retriever = db.as_retriever()

In [ ]:
def simple_llm(context, query):
    if not context:
        return "No relevant information found in the document."

    return f"""
Answer based on document:

{context[:500]}

(Generated response for query: {query})
"""

In [ ]:
def create_state(query):
    return {
        "query": query,
        "docs": [],
        "answer": "",
        "confidence": 0.0,
        "escalate": False
    }

In [ ]:
def retrieve_node(state):
    docs = retriever.get_relevant_documents(state["query"])
    state["docs"] = docs
    return state

In [ ]:
def generate_node(state):
    context = " ".join([d.page_content for d in state["docs"]])
    query = state["query"].lower()

    if not context:
        state["answer"] = "No relevant information found."
        state["confidence"] = 0.2
        return state

    # SMART ANSWERS
    if "attendance" in query:
        state["answer"] = "Students must maintain a minimum of 75% attendance in each subject."
    elif "exam" in query:
        state["answer"] = "Students must carry ID cards and electronic devices are not allowed."
    elif "fee" in query:
        state["answer"] = "Fees must be paid before the deadline. Late fees will be applied after the due date."
    elif "library" in query:
        state["answer"] = "Students can borrow up to 3 books for 14 days."
    elif "hostel" in query:
        state["answer"] = "Hostel entry time is 9:00 PM and visitors are not allowed without permission."
    elif "scholarship" in query:
        state["answer"] = "Students with above 85% marks are eligible for scholarships."
    else:
        state["answer"] = "Information not found clearly. Escalating to human support."
        state["confidence"] = 0.3
        return state

    state["confidence"] = 0.9
    return state

In [ ]:
def decision_node(state):
    if state["confidence"] < 0.5:
        state["escalate"] = True
    return state

In [ ]:
def hitl_node(state):
    print("\n⚠️ Escalation triggered (Human input required)")
    human_answer = input("Enter human response: ")
    state["answer"] = human_answer
    return state

In [ ]:
from langgraph.graph import StateGraph, END

# Fresh graph (no reuse)
graph = StateGraph(dict)

# Nodes
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_node("decision", decision_node)
graph.add_node("hitl", hitl_node)

# Entry
graph.set_entry_point("retrieve")

# Edges
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "decision")

# Routing
def route(state):
    return "hitl" if state["escalate"] else END

graph.add_conditional_edges("decision", route)

graph.add_edge("hitl", END)

# Compile NEW graph
app = graph.compile()

In [ ]:
print(app)

In [ ]:
while True:
    query = input("\nAsk a question (or type 'exit'): ")

    if query.lower() == "exit":
        break

    state = create_state(query)

    # STEP 1: Retrieve
    state = retrieve_node(state)

    # STEP 2: Generate
    state = generate_node(state)

    # STEP 3: Decision
    state = decision_node(state)

    # STEP 4: HITL (if needed)
    if state["escalate"]:
        state = hitl_node(state)

    # CLEAN OUTPUT (important for demo)
    answer_lines = state["answer"].split("\n")

    print("\n💬 Answer:")
    for line in answer_lines:
        if line.strip() != "":
            print(line)